# Weekly KPI EDA

Phase 1 notebook for the Yoobic Insight prototype. This notebook is intentionally read-only: it validates the source table, confirms the sales identity, documents coverage, surfaces the expected `Store_G` 2025 week 21 violation, and locks the robust constants later phases will reuse.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "AGENTS.md").exists() and (candidate / "docs").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root from notebook execution context.")


ROOT = find_repo_root(Path.cwd().resolve())
DATA_PATH = ROOT / "data" / "raw" / "practical-test-dataset-weekly-kpi.xlsx"
IDENTITY_ABS_TOLERANCE = 1e-9
MODIFIED_ZSCORE_THRESHOLD = 3.5

df = pd.read_excel(DATA_PATH)
df.head()


## 1. Schema

Validate the table grain and basic completeness before any KPI derivation.


In [ ]:
schema_summary = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "null_count": df.isna().sum(),
    }
)

key_duplicates = int(df.duplicated(["Store Name", "Year", "Week"]).sum())
shape_summary = pd.DataFrame(
    [
        {"metric": "rows", "value": len(df)},
        {"metric": "columns", "value": df.shape[1]},
        {"metric": "stores", "value": df["Store Name"].nunique()},
        {"metric": "years", "value": ", ".join(map(str, sorted(df["Year"].unique())))},
        {"metric": "duplicate_store_year_week_keys", "value": key_duplicates},
    ]
)

display(shape_summary)
display(schema_summary)


## 2. Identity Check

Confirm that `net_sales = traffic × conversion_rate × units_per_txn × avg_selling_price` holds up to floating-point precision and lock the tolerance for later phases.


In [ ]:
analysis = df.copy()
analysis["conversion_rate"] = analysis["gross_transactions"] / analysis["traffic"]
analysis["units_per_txn"] = analysis["gross_quantity"] / analysis["gross_transactions"]
analysis["avg_selling_price"] = analysis["net_sales"] / analysis["gross_quantity"]
analysis["reconstructed_net_sales"] = (
    analysis["traffic"]
    * analysis["conversion_rate"]
    * analysis["units_per_txn"]
    * analysis["avg_selling_price"]
)
analysis["identity_abs_drift"] = (analysis["net_sales"] - analysis["reconstructed_net_sales"]).abs()
analysis["identity_rel_drift"] = analysis["identity_abs_drift"] / analysis["net_sales"].abs()

identity_summary = pd.DataFrame(
    [
        {"metric": "max_abs_drift", "value": analysis["identity_abs_drift"].max()},
        {"metric": "p99_abs_drift", "value": analysis["identity_abs_drift"].quantile(0.99)},
        {"metric": "max_rel_drift", "value": analysis["identity_rel_drift"].max()},
        {"metric": "rows_gt_1e-9", "value": int((analysis["identity_abs_drift"] > IDENTITY_ABS_TOLERANCE).sum())},
        {"metric": "identity_abs_tolerance", "value": IDENTITY_ABS_TOLERANCE},
    ]
)

display(identity_summary)
assert analysis["identity_abs_drift"].max() <= IDENTITY_ABS_TOLERANCE


## 3. Coverage

Document the available week range per year and per store-year so later YoY logic uses a like-for-like week intersection.


In [ ]:
coverage_by_year = df.groupby("Year")["Week"].agg(["min", "max", "nunique"]).reset_index()
coverage_by_store_year = df.groupby(["Store Name", "Year"])["Week"].agg(["min", "max", "nunique"]).reset_index()

weeks_2024 = set(df.loc[df["Year"] == 2024, "Week"])
weeks_2025 = set(df.loc[df["Year"] == 2025, "Week"])
global_intersection = sorted(weeks_2024 & weeks_2025)

intersection_rows = []
for store, group in df.groupby("Store Name"):
    store_weeks_2024 = set(group.loc[group["Year"] == 2024, "Week"])
    store_weeks_2025 = set(group.loc[group["Year"] == 2025, "Week"])
    overlap = sorted(store_weeks_2024 & store_weeks_2025)
    intersection_rows.append(
        {
            "Store Name": store,
            "overlap_week_count": len(overlap),
            "overlap_week_min": overlap[0],
            "overlap_week_max": overlap[-1],
        }
    )

display(coverage_by_year)
display(coverage_by_store_year)
display(pd.DataFrame(intersection_rows))
print(f"Global week intersection: {global_intersection[0]}..{global_intersection[-1]} ({len(global_intersection)} weeks)")


## 4. Store_G Week 21 Violation

Surface the known data-quality exception manually so it becomes an explicit validation target in Phase 2.


In [ ]:
store_g_w21 = df[(df["Store Name"] == "Store_G") & (df["Year"] == 2025) & (df["Week"] == 21)].copy()
store_g_w21["gross_transactions_minus_traffic"] = store_g_w21["gross_transactions"] - store_g_w21["traffic"]
store_g_w21["conversion_rate"] = store_g_w21["gross_transactions"] / store_g_w21["traffic"]
store_g_w21["units_per_txn"] = store_g_w21["gross_quantity"] / store_g_w21["gross_transactions"]
store_g_w21["avg_selling_price"] = store_g_w21["net_sales"] / store_g_w21["gross_quantity"]

display(store_g_w21)
assert len(store_g_w21) == 1
assert bool((store_g_w21["gross_transactions"] > store_g_w21["traffic"]).iloc[0])


## 5. Network Distributions

Use robust statistics across the weekly panel to establish medians, MADs, and the default modified z-score threshold for anomaly scoring in later phases.


In [ ]:
weekly = analysis.copy()
weekly["avg_txn_value"] = weekly["net_sales"] / weekly["gross_transactions"]
weekly["revenue_per_visitor"] = weekly["net_sales"] / weekly["traffic"]

metrics = [
    "traffic",
    "gross_transactions",
    "gross_quantity",
    "net_sales",
    "conversion_rate",
    "units_per_txn",
    "avg_selling_price",
    "avg_txn_value",
    "revenue_per_visitor",
]

distribution_rows = []
for metric in metrics:
    values = weekly[metric].to_numpy(dtype=float)
    median = float(np.median(values))
    mad = float(np.median(np.abs(values - median)))
    distribution_rows.append(
        {
            "metric": metric,
            "median": median,
            "mad": mad,
            "q1": float(np.quantile(values, 0.25)),
            "q3": float(np.quantile(values, 0.75)),
            "min": float(values.min()),
            "max": float(values.max()),
            "modified_zscore_threshold": MODIFIED_ZSCORE_THRESHOLD,
        }
    )

distribution_summary = pd.DataFrame(distribution_rows).round(4)
display(distribution_summary)
